# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohinaRustamova/lyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## 1. My Lane and Why

**Lane: Refresh / Content Opportunity Scoring**

I'm choosing this lane because my Week 1–2 work (Notebooks 01–02) already
engaged directly with its core mechanics. In Notebook 02, I built a
hand-written baseline rule (stale × visible pages) and compared it against
a learned decision tree using Precision@K — exactly the workflow this lane
is built around. I also personally encountered the leakage trap this lane
warns about firsthand: feeding `trend_pct` into the model produced a
suspicious Precision@50 of 1.000, which taught me why this lane restricts
features to observable-only signals.

This lane also matches the entire starter pipeline (`run_all.py`), so I can
build forward from a foundation I already understand rather than starting a
new methodology from scratch.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The Question: Decision, Action, Cost

**Research question:** Which content pages should be reviewed first for
refresh, given limited reviewer capacity?

**Unit of analysis:** One content page (`content_id`), evaluated over a
90-day observation window.

**Decision this improves:** Without this, a reviewer would find candidate
pages by manual browsing or ad hoc judgment. This ranks the full inventory
so limited reviewer time goes to the highest-opportunity pages first.

**Action taken from the output:** A content reviewer opens the top-K pages
from the ranked queue and evaluates each for refresh, expansion, or
monitoring, guided by the reason codes attached to each page's score.

**Cost of a wrong recommendation:**
- **False positive** (flagged as worth reviewing, but wasn't really an
  issue): wastes reviewer time — low-to-moderate cost, since a human still
  makes the final call before any action is taken.
- **False negative** (a genuinely declining page never surfaces in the
  queue): higher cost — a real problem goes unaddressed and may keep losing
  traffic silently, since no one is looking for it manually.

This asymmetry is why Precision@K (getting the top of the list right)
matters more here than overall accuracy — a reviewer only ever looks at the
top of the queue, never the full ranked list.

**Why data/ML can help at all:** A fixed rule (e.g. "stale AND visible") is
transparent but rigid. Notebook 02 showed the hand rule beats a simple tree
at the very top of the ranking (Precision@20: 0.90 vs. 0.55) but loses
further down (held-out Precision@50: 0.65–0.68 for the tree vs. 0.68 for
the rule). This suggests neither a pure rule nor a naive model alone
captures the full picture — there's likely room for a better-tuned model to
combine multiple weak signals that would be impractical for a human to
manually track across thousands of pages.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [9]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MohinaRustamova/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
print("Contents:", os.listdir())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check repo root"
print("Ready.")

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship
Contents: ['requirements.txt', 'outputs', 'data', '.github', '.gitignore', 'SETUP.md', '.git', 'docs', 'skills', 'README.md', 'submission', 'notebooks', 'AGENTS.md', 'work', 'CLAUDE.md', 'GUIDE.md', 'scripts', 'LICENSE', 'DATA_USE.md']
Ready.


In [10]:
import pandas as pd
import json
import sys
!{sys.executable} scripts/run_all.py
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

declining_rate = (df["trend_direction"] == "down").mean()
print(f"Share of pages currently in decline: {declining_rate:.1%}")

res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline rule Precision@50: {base:.3f}  |  Random forest Precision@50: {rf:.3f}")
print(f"Lift: {rf/base:.1f}x")

visible = df[df["impressions_90d"] >= 100]
counts = visible.groupby("position_tier")["ctr"].count()
print("\nPages per position tier (adequate-sample context):")
print(counts.to_string())


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/flyrank-ml-internship/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/flyrank-ml-internship/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /c

**What these numbers show:**

1. Over half the pages (54.2%) are currently labeled "declining" — a large
   enough pool that prioritization genuinely matters; reviewers can't check
   everything.
2. The random forest beats the hand rule by 3.1x on Precision@50 (0.740 vs.
   0.240) on the starter slice — real evidence that a learned model adds
   value over a fixed rule, at least at this scale.
3. Sample sizes vary sharply by position tier — from 533 pages (top_3) to
   8,633 (page_1) — a direct reminder from my Notebook 01 work that any
   scoring system I build needs minimum-volume filters before trusting a
   reason code.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful Words: What I Can and Can't Claim

**What my work CAN say:**
- This is decision-support, not proof of causation. A high-scoring page is
  a candidate worth human review — not a guaranteed win if refreshed.
- Any observed pattern (e.g. which pages under-capture clicks or traffic at
  a given position) is directional — "this suggests," not "this proves."
- Results are based on a 30,000-row anonymized starter slice and may not
  hold at full warehouse scale (78.8M rows) without re-validation.
- If a learned model beats a hand-written rule on Precision@K, that's real,
  measurable evidence the model adds value on *this* validation split — not
  proof it will always outperform simpler rules in every context.

**What my work will NEVER claim:**
- That refreshing a flagged page will *cause* a recovery in traffic or
  rankings — proving that would require a real experiment (e.g. an A/B
  test on actual refreshed vs. non-refreshed pages), which this data alone
  cannot provide.
- Any claim about Google's ranking algorithm, its specific factors, or how
  it works internally. Nothing in this data reveals algorithm behavior —
  only observed outcomes (impressions, clicks, position) after the fact.
- That the current label (`is_declining_label`, based on `trend_direction
  == "down"`) reflects ground truth about page quality — it's a same-window
  proxy label, not a validated future outcome. I plan to move toward a
  stronger future-window label (prior 90-day features → next 30-day
  outcome) as the lane guide recommends.
- That a page flagged as "declining" is definitely a real decline, rather
  than consolidation (traffic shifted to a sibling page), seasonality, or
  simple noise — distinguishing these requires deeper checks I haven't yet
  built.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.